<a href="https://colab.research.google.com/github/madaam99/EnergyGermany/blob/main/SMARD_DataAcquisition_Energy_Germany_Weekly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Getting the data

In [1]:
import requests
from datetime import datetime, timedelta
import os

In [2]:
def smard_download(payload, filename):
    """
    Downloads SMARD CSV data using the given payload and saves it to filename.
    """
    url = "https://www.smard.de/nip-download-manager/nip/download/market-data"

    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code != 200:
        raise RuntimeError(f"SMARD returned status {response.status_code}")

    text = response.text

    # SMARD sometimes returns a CSV that only contains "Keine Daten"
    if "Keine Daten" in text:
        print(f"No data available for: {filename}")
    else:
        print(f"Download OK for: {filename}")

    with open(filename, "wb") as f:
        f.write(response.content)


In [3]:
today = datetime.today()
five_years_ago = today - timedelta(days=5*365)

timestamp_to = int(today.timestamp() * 1000)
timestamp_from = int(five_years_ago.timestamp() * 1000)

payload_installed_energy = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                3004073,3004076,3004072,3004074,3004075,
                3000186,3000188,3000189,3000194,3000198,
                3003792,3000207
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "month"
        }
    ]
}

payload_energy_production = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                1001224,1004066,1004067,1004068,
                1001223,1004069,1004071,1004070,
                1001226,1001228,1001227,1001225
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "week"
        }
    ]
}

payload_energy_consumption = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                5000410,5004387,5005140,5004359
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "week"
        }
    ]
}

smard_download(payload_installed_energy, "installed_energy.csv")
smard_download(payload_energy_production, "energy_production.csv")
smard_download(payload_energy_consumption, "energy_consumption.csv")

print("All downloads completed.")

Download OK for: installed_energy.csv
Download OK for: energy_production.csv
Download OK for: energy_consumption.csv
All downloads completed.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Installierte Erzeugungsleistung

In [5]:
instEnergy = pd.read_csv("/content/installed_energy.csv", delimiter=";")
instEnergy.head()

,Datum von,Datum bis,Biomasse [MW] Berechnete Auflösungen,Wasserkraft [MW] Berechnete Auflösungen,Wind Offshore [MW] Berechnete Auflösungen,Wind Onshore [MW] Berechnete Auflösungen,Photovoltaik [MW] Berechnete Auflösungen,Sonstige Erneuerbare [MW] Berechnete Auflösungen,Kernenergie [MW] Berechnete Auflösungen,Braunkohle [MW] Berechnete Auflösungen,Steinkohle [MW] Berechnete Auflösungen,Erdgas [MW] Berechnete Auflösungen,Pumpspeicher [MW] Berechnete Auflösungen,Sonstige Konventionelle [MW] Berechnete Auflösungen
0,01.01.2021,01.02.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
1,01.02.2021,01.03.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
2,01.03.2021,01.04.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
3,01.04.2021,01.05.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
4,01.05.2021,01.06.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-


In [6]:
instEnergy = instEnergy.rename(columns=lambda x: x.replace('Berechnete Auflösungen', '').strip())

instEnergy = instEnergy.replace("-", np.nan).fillna(0)

numeric_cols = instEnergy.columns.drop(['Datum von', 'Datum bis'])

for col in numeric_cols:
    instEnergy[col] = instEnergy[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    instEnergy[col] = pd.to_numeric(instEnergy[col])

instEnergy.insert(2, "Gesamtkapazität [MW]", instEnergy.drop(columns=["Datum von", "Datum bis"]).sum(axis=1))

instEnergy["Datum von"] = pd.to_datetime(instEnergy["Datum von"], format="%d.%m.%Y")
instEnergy["Datum bis"] = pd.to_datetime(instEnergy["Datum bis"], format="%d.%m.%Y")

instEnergy = instEnergy.loc[~(instEnergy.iloc[:, 2:] == 0).all(axis=1)]

instEnergy.head()

/tmp/ipython-input-1366071424.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  instEnergy = instEnergy.replace("-", np.nan).fillna(0)


,Datum von,Datum bis,Gesamtkapazität [MW],Biomasse [MW],Wasserkraft [MW],Wind Offshore [MW],Wind Onshore [MW],Photovoltaik [MW],Sonstige Erneuerbare [MW],Kernenergie [MW],Braunkohle [MW],Steinkohle [MW],Erdgas [MW],Pumpspeicher [MW],Sonstige Konventionelle [MW]
0,2021-01-01,2021-02-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0,0.0,0.0,31942.0,0.0,0.0
1,2021-02-01,2021-03-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0,0.0,0.0,31942.0,0.0,0.0
2,2021-03-01,2021-04-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0,0.0,0.0,31942.0,0.0,0.0
3,2021-04-01,2021-05-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0,0.0,0.0,31942.0,0.0,0.0
4,2021-05-01,2021-06-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0,0.0,0.0,31942.0,0.0,0.0


In [7]:
instEnergy_mwh = instEnergy.copy()

instEnergy_mwh['Datum von'] = pd.to_datetime(instEnergy_mwh['Datum von'])
instEnergy_mwh['Datum bis'] = pd.to_datetime(instEnergy_mwh['Datum bis'])

instEnergy_mwh['Duration_Hours'] = (instEnergy_mwh['Datum bis'] - instEnergy_mwh['Datum von']).dt.total_seconds() / 3600

mw_columns = [col for col in instEnergy_mwh.columns if '[MW]' in col]

for col in mw_columns:
    instEnergy_mwh[col] = instEnergy_mwh[col] * instEnergy_mwh['Duration_Hours']

instEnergy_mwh.columns = [col.replace('[MW]', '[MWh]') if '[MW]' in col else col for col in instEnergy_mwh.columns]

instEnergy_mwh.drop(columns=['Duration_Hours'], inplace=True)

instEnergy_mwh.head()

,Datum von,Datum bis,Gesamtkapazität [MWh],Biomasse [MWh],Wasserkraft [MWh],Wind Offshore [MWh],Wind Onshore [MWh],Photovoltaik [MWh],Sonstige Erneuerbare [MWh],Kernenergie [MWh],Braunkohle [MWh],Steinkohle [MWh],Erdgas [MWh],Pumpspeicher [MWh],Sonstige Konventionelle [MWh]
0,2021-01-01,2021-02-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0
1,2021-02-01,2021-03-01,97373472.0,5513088.0,0.0,0.0,36519840.0,33875520.0,0.0,0.0,0.0,0.0,21465024.0,0.0,0.0
2,2021-03-01,2021-04-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0
3,2021-04-01,2021-05-01,104328720.0,5906880.0,0.0,0.0,39128400.0,36295200.0,0.0,0.0,0.0,0.0,22998240.0,0.0,0.0
4,2021-05-01,2021-06-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0


In [8]:
instEnergy.to_csv("processedInstalledEnergy", index=False)

Realisierte Stromerzeugung

In [9]:
erzeugung = pd.read_csv("/content/energy_production.csv", delimiter=";")
erzeugung.head()

,Datum von,Datum bis,Biomasse [MWh] Berechnete Auflösungen,Wasserkraft [MWh] Berechnete Auflösungen,Wind Offshore [MWh] Berechnete Auflösungen,Wind Onshore [MWh] Berechnete Auflösungen,Photovoltaik [MWh] Berechnete Auflösungen,Sonstige Erneuerbare [MWh] Berechnete Auflösungen,Kernenergie [MWh] Berechnete Auflösungen,Braunkohle [MWh] Berechnete Auflösungen,Steinkohle [MWh] Berechnete Auflösungen,Erdgas [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Sonstige Konventionelle [MWh] Berechnete Auflösungen
0,14.12.2020,21.12.2020,"778.769,25","228.452,00","840.723,00","2.282.851,75","208.337,50","36.148,00","1.319.474,75","2.115.493,25","969.267,25","1.602.918,75","196.815,50","267.121,75"
1,21.12.2020,28.12.2020,"769.236,00","225.564,50","659.089,25","3.178.818,00","142.913,50","33.542,00","1.295.255,75","1.097.438,75","400.630,25","1.049.778,75","196.195,00","251.109,50"
2,28.12.2020,04.01.2021,"772.607,00","212.520,50","337.175,00","1.674.185,00","156.765,25","37.336,00","1.330.677,00","1.699.151,00","597.597,00","1.354.440,50","167.296,75","271.930,25"
3,04.01.2021,11.01.2021,"771.324,00","219.139,50","483.398,50","1.066.496,00","78.002,50","35.675,25","1.340.445,00","2.409.738,25","1.519.282,50","1.961.371,00","205.910,75","276.376,00"
4,11.01.2021,18.01.2021,"763.837,50","216.574,25","588.842,25","2.420.537,75","117.714,75","34.127,00","1.331.288,50","2.331.575,25","1.261.929,50","1.754.224,50","182.043,75","266.634,25"


In [10]:
erzeugung = erzeugung.rename(columns=lambda x: x.replace('Berechnete Auflösungen', '').strip())

erzeugung = erzeugung.replace("-", np.nan).fillna(0)

numeric_cols = erzeugung.columns.drop(['Datum von', 'Datum bis'])

for col in numeric_cols:
    erzeugung[col] = erzeugung[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    erzeugung[col] = pd.to_numeric(erzeugung[col])

erzeugung.insert(2, "Gesamt [MWh]", erzeugung.drop(columns=["Datum von", "Datum bis"]).sum(axis=1))

erzeugung["Datum von"] = pd.to_datetime(erzeugung["Datum von"], format="%d.%m.%Y")
erzeugung["Datum bis"] = pd.to_datetime(erzeugung["Datum bis"], format="%d.%m.%Y")

erzeugung = erzeugung.loc[~(erzeugung.iloc[:, 2:] == 0).all(axis=1)]

erzeugung.head()

,Datum von,Datum bis,Gesamt [MWh],Biomasse [MWh],Wasserkraft [MWh],Wind Offshore [MWh],Wind Onshore [MWh],Photovoltaik [MWh],Sonstige Erneuerbare [MWh],Kernenergie [MWh],Braunkohle [MWh],Steinkohle [MWh],Erdgas [MWh],Pumpspeicher [MWh],Sonstige Konventionelle [MWh]
0,2020-12-14,2020-12-21,10846372.75,778769.25,228452.00,840723.00,2282851.75,208337.50,36148.00,1319474.75,2115493.25,969267.25,1602918.75,196815.50,267121.75
1,2020-12-21,2020-12-28,9299571.25,769236.00,225564.50,659089.25,3178818.00,142913.50,33542.00,1295255.75,1097438.75,400630.25,1049778.75,196195.00,251109.50
2,2020-12-28,2021-01-04,8611681.25,772607.00,212520.50,337175.00,1674185.00,156765.25,37336.00,1330677.00,1699151.00,597597.00,1354440.50,167296.75,271930.25
3,2021-01-04,2021-01-11,10367159.25,771324.00,219139.50,483398.50,1066496.00,78002.50,35675.25,1340445.00,2409738.25,1519282.50,1961371.00,205910.75,276376.00
4,2021-01-11,2021-01-18,11269329.25,763837.50,216574.25,588842.25,2420537.75,117714.75,34127.00,1331288.50,2331575.25,1261929.50,1754224.50,182043.75,266634.25


In [11]:
erzeugung.to_csv("processedErzeugung", index=False)

Realisierter Stromverbrauch

In [12]:
verbrauch = pd.read_csv("/content/energy_consumption.csv", delimiter=";")
verbrauch.head()

,Datum von,Datum bis,Netzlast [MWh] Berechnete Auflösungen,Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Residuallast [MWh] Berechnete Auflösungen
0,14.12.2020,21.12.2020,"10.021.246,50","10.255.350,25","234.103,75","6.689.334,25"
1,21.12.2020,28.12.2020,"8.653.659,75","8.898.908,75","245.249,00","4.672.839,00"
2,28.12.2020,04.01.2021,"8.408.100,25","8.648.705,25","240.605,00","6.239.975,00"
3,04.01.2021,11.01.2021,"10.098.335,25","10.307.229,75","208.894,50","8.470.298,38"
4,11.01.2021,18.01.2021,"10.818.243,25","11.052.059,00","233.815,75","7.691.148,50"


In [13]:
verbrauch = verbrauch.rename(columns=lambda x: x.replace('Berechnete Auflösungen', '').strip())

verbrauch = verbrauch.replace("-", np.nan).fillna(0)

verbrauch['Datum von'] = pd.to_datetime(verbrauch['Datum von'], format='%d.%m.%Y')
verbrauch['Datum bis'] = pd.to_datetime(verbrauch['Datum bis'], format='%d.%m.%Y')

num_col = verbrauch.columns.drop(['Datum von', 'Datum bis'])

for col in num_col:
    verbrauch[col] = verbrauch[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

verbrauch = verbrauch.loc[~(verbrauch.iloc[:, 2:] == 0).all(axis=1)]
verbrauch.insert(2, "Gesamtlast [MWh]", verbrauch[["Netzlast [MWh]", "Pumpspeicher [MWh]", "Residuallast [MWh]"]].sum(axis=1))

verbrauch.head()

,Datum von,Datum bis,Gesamtlast [MWh],Netzlast [MWh],Netzlast inkl. Pumpspeicher [MWh],Pumpspeicher [MWh],Residuallast [MWh]
0,2020-12-14,2020-12-21,16944684.50,10021246.50,10255350.25,234103.75,6689334.25
1,2020-12-21,2020-12-28,13571747.75,8653659.75,8898908.75,245249.00,4672839.00
2,2020-12-28,2021-01-04,14888680.25,8408100.25,8648705.25,240605.00,6239975.00
3,2021-01-04,2021-01-11,18777528.13,10098335.25,10307229.75,208894.50,8470298.38
4,2021-01-11,2021-01-18,18743207.50,10818243.25,11052059.00,233815.75,7691148.50


In [14]:
(verbrauch['Netzlast inkl. Pumpspeicher [MWh]'] == (verbrauch['Netzlast [MWh]'] + verbrauch['Pumpspeicher [MWh]'])).all()

np.False_

In [15]:
verbrauch = verbrauch.loc[~(verbrauch['Netzlast inkl. Pumpspeicher [MWh]'] != (verbrauch['Netzlast [MWh]'] + verbrauch['Pumpspeicher [MWh]']))]
verbrauch.tail()

,Datum von,Datum bis,Gesamtlast [MWh],Netzlast [MWh],Netzlast inkl. Pumpspeicher [MWh],Pumpspeicher [MWh],Residuallast [MWh]
252,2025-10-13,2025-10-20,15827562.42,9118271.00,9325939.12,207668.12,6501623.30
253,2025-10-20,2025-10-27,12463648.45,9418648.82,9670054.82,251406.00,2793593.63
254,2025-10-27,2025-11-03,12984703.90,9208328.27,9411613.23,203284.96,3573090.67
255,2025-11-03,2025-11-10,16334226.60,9551540.40,9759560.72,208020.32,6574665.88
256,2025-11-10,2025-11-17,15669831.90,9572060.69,9752828.91,180768.22,5917002.99


In [16]:
verbrauch.to_csv("processedVerbrauch", index=False)